# PermaWatch: Exploratory Data Analysis & Model Training Pipeline
**A Satellite Imagery-Based Permafrost Detection and Monitoring System Using Machine Learning**

This notebook walks through the end-to-end scientific methodology:
1. Loading the processed Arctic-Boreal training dataset (derived from ESA CCI Permafrost v5.0).
2. Examining environmental predictors: Sentinel-2 optical bands, NDVI, NDWI, NDSI, elevation, and terrain slope.
3. Exploratory Data Analysis (EDA) of Arctic-boreal environmental predictors.
4. Spatial-block cross-validation (`GroupShuffleSplit`) to eliminate spatial autocorrelation data leakage.
5. Evaluating THREE machine learning models: **Random Forest**, **XGBoost**, and **Support Vector Machine (SVM)**.
6. Model comparison, ROC curves, confusion matrices, and feature importance analysis.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully!')

## 1. Load Processed Training Dataset
We load the curated dataset containing 29,294 observations across 5 distinct latitudinal blocks in Alaska.

In [2]:
data_path = '../data/processed/permafrost_training_dataset.parquet'
df = pd.read_parquet(data_path)
print(f'Dataset Shape: {df.shape}')
df.head()

## 2. Target Class Distribution & Regional Breakdown

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='permafrost_class', ax=ax1, palette=['#38a169', '#e53e3e'])
ax1.set_title('Permafrost Dominance Class Balance')
ax1.set_xticklabels(['Non-Permafrost (<50%)', 'Permafrost-Dominant (>=50%)'])

sns.countplot(data=df, y='region_name', hue='permafrost_class', ax=ax2, palette=['#38a169', '#e53e3e'])
ax2.set_title('Permafrost Distribution Across Latitudinal Zones')
plt.tight_layout()
plt.show()

## 3. Topographic and Spectral Predictor Analysis
Let us inspect how elevation, slope, and vegetation/moisture indices relate to permafrost occurrence.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df, x='permafrost_class', y='elevation', ax=axes[0,0], palette=['#38a169', '#e53e3e'])
axes[0,0].set_title('Elevation (m) by Class')
axes[0,0].set_xticklabels(['Non-Permafrost', 'Permafrost'])

sns.boxplot(data=df, x='permafrost_class', y='slope', ax=axes[0,1], palette=['#38a169', '#e53e3e'])
axes[0,1].set_title('Terrain Slope (deg) by Class')
axes[0,1].set_xticklabels(['Non-Permafrost', 'Permafrost'])

sns.boxplot(data=df, x='permafrost_class', y='ndvi', ax=axes[1,0], palette=['#38a169', '#e53e3e'])
axes[1,0].set_title('NDVI (Vegetation Index) by Class')
axes[1,0].set_xticklabels(['Non-Permafrost', 'Permafrost'])

sns.boxplot(data=df, x='permafrost_class', y='ndsi', ax=axes[1,1], palette=['#38a169', '#e53e3e'])
axes[1,1].set_title('NDSI (Snow/Ice Index) by Class')
axes[1,1].set_xticklabels(['Non-Permafrost', 'Permafrost'])
plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [5]:
features = ['latitude', 'longitude', 'elevation', 'slope', 'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'ndwi', 'ndsi']
plt.figure(figsize=(11, 8))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Matrix of Environmental Features')
plt.show()

## 5. Model Training & Spatial-Block Evaluation Results
We display the rigorous spatial evaluation metrics from `outputs/metrics/model_comparison.csv`.

In [6]:
metrics_df = pd.read_csv('../outputs/metrics/model_comparison.csv')
print('=== Three-Model Spatial-Block Validation Results ===')
metrics_df

## 6. Single-Point Inference Test
We test inference on sample points representing Arctic Foothills vs Interior Fairbanks.

In [7]:
sys.path.append('..')
from src.predict import predict_point

sample_pt = df[df['region_name'] == 'North_Slope_Prudhoe'].iloc[0].to_dict()
res = predict_point(sample_pt, model_name='Random Forest')
print('Prediction for North Slope Point:')
print('  Predicted Class:', res['class_label'])
print('  Likelihood Score:', res['likelihood_percentage'])
print('  Risk Category:', res['risk_level'])